In [ ]:
# Install transformers if not already installed
# !pip install transformers

import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load IndoBERTweet tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")
model = AutoModel.from_pretrained("indolem/indobertweet-base-uncased")

# Load your data
df = pd.read_json('fetched_data_final.json')
X = df['text'].tolist()
y = df['label'].astype(int).tolist()

In [ ]:


# Function to get sentence embeddings
def get_bertweet_embeddings(texts, tokenizer, model, max_length=128, batch_size=16):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            outputs = model(**encoded)
            # Use [CLS] token embedding as sentence representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

In [ ]:

# Get IndoBERTweet embeddings for all texts
X_embeddings = get_bertweet_embeddings(X, tokenizer, model)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42, stratify=y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Get IndoBERTweet embeddings for all texts
X_embeddings = get_bertweet_embeddings(X, tokenizer, model)
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
all_reports = []
all_y_true = []
all_y_pred = []
cv_scores = []
fold_reports = []
fold = 0

for fold, (train_idx, test_idx) in enumerate(skf.split(X_embeddings, y)):
    X_train, X_test = X_embeddings[train_idx], X_embeddings[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]

    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    print(f"Fold {fold+1}")
    print(classification_report(y_test, y_pred, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    accu = accuracy_score(y_test, y_pred)
    cv_scores.append(accu)